In [ ]:

import json,re,sys,subprocess
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','-q','nibabel','beautifulsoup4'],check=True)
import requests, nibabel, numpy as np, hashlib
from bs4 import BeautifulSoup
cache=Path('/tmp/cshaper_audit')
cache.mkdir(exist_ok=True)
def listing(folder):
    response=requests.get('https://drive.google.com/embeddedfolderview',params={'id':folder},timeout=40)
    response.raise_for_status()
    entries={}
    for anchor in BeautifulSoup(response.text,'html.parser').select('a'):
        title=anchor.select_one('.flip-entry-title')
        match=re.search(r'/(?:folders|d)/([\w-]+)',anchor.get('href',''))
        if title and match:
            entries[title.get_text(strip=True)]=match[1]
    return entries

def frames(entries, sample, suffix):
    pattern=rf'{sample}_(\d+)_{suffix}\.nii\.gz'
    return {int(m[1]): key for name,key in entries.items() if (m:=re.fullmatch(pattern,name))}

def fetch(key):
    path=cache/f'{key}.nii.gz'
    if not path.exists():
        from requests.adapters import HTTPAdapter
        from urllib3.util.retry import Retry
        with requests.Session() as session:
            session.mount('https://', HTTPAdapter(max_retries=Retry(total=3, backoff_factor=2,
                status_forcelist=(429, 500, 502, 503, 504))))
            with session.get('https://drive.usercontent.google.com/download',
                             params=dict(id=key, export='download', confirm='t'),
                             stream=True, timeout=(20, 120)) as response:
                response.raise_for_status()
                chunks = response.iter_content(1024 * 1024)
                head = next(chunks, b'')
                if not head.startswith(bytes((31, 139))):
                    raise ValueError(f'Drive did not serve a gzip file for {key}')
                staging = path.with_suffix('.part')
                with staging.open('wb') as output:
                    output.write(head)
                    for chunk in chunks:
                        output.write(chunk)
                staging.replace(path)
    return nibabel.load(path), hashlib.sha256(path.read_bytes()).hexdigest()


def download_text(key):
    r=requests.get('https://drive.usercontent.google.com/download',
                   params=dict(id=key,export='download',confirm='t'),timeout=60)
    r.raise_for_status()
    return r.text


import csv, io
from collections import Counter
names={int(k):v for k,v in list(csv.reader(io.StringIO(download_text(
    '1zBqmyu9kZJ9NMM1YjHzQrIGftd_Q4BJk'))))[1:] if k.isdigit()}
raw=listing('151pBI7d_JfkOz4eQI4R4F2u_cKtlftBi')
labels=listing('1m977Ong7XR9KZPWsRdLr03otf7S_J9dx')
report={'method':'SegNuc native IDs resolved via published name dictionary. SegCell overlap is a diagnostic only.', 'samples':{}}
output=Path('/kaggle/working/lineage_audit.json')

def match_frame(nucleus_file, cell_file):
    nuc, nuc_sha=fetch(nucleus_file)
    cell, cell_sha=fetch(cell_file)
    a = np.asarray(nuc.dataobj)
    direct = np.asarray(cell.dataobj)
    if a.shape != direct.shape:
        raise ValueError('Voxel array dimensions differ')
    ids = np.unique(a[a > 0])
    named = {int(key): names[int(key)] for key in ids if key in names}

    def evaluate(b):
        valid=a>0
        pairs,counts=np.unique(np.column_stack((a[valid],b[valid])),axis=0,return_counts=True)
        matches=[]
        for nucleus in np.unique(pairs[:,0]):
            mask=pairs[:,0]==nucleus
            candidates=pairs[mask,1]
            weights=counts[mask]
            best=int(np.argmax(weights))
            label=int(candidates[best])
            overlap=float(weights[best]/weights.sum())
            if label in names and overlap>=0.75:
                matches.append((int(nucleus),names[label]))
        from collections import Counter
        uses=Counter(name for _,name in matches)
        unique={key:name for key,name in matches if uses[name]==1}
        return unique

    index = evaluate(direct)
    overlaps = Counter()
    for key in ids:
        values = direct[a == key]
        labels, counts = np.unique(values, return_counts=True)
        if not len(labels):
            continue
        best = int(np.argmax(counts))
        region = int(labels[best])
        if counts[best] / len(values) >= 0.75 and region in names:
            overlaps[region] += 1
    return named, dict(nuclei=len(ids), mapped=len(named),
        index_mapped=len(index), matching_names=sum(named.get(k)==v for k,v in index.items()),
        shared_cell_regions=sum(count>1 for count in overlaps.values()),
        nucleus_affine=nuc.affine.tolist(), cell_affine=cell.affine.tolist(),
        nucleus_units=nuc.header.get_xyzt_units()[0], cell_units=cell.header.get_xyzt_units()[0],
        nucleus_axes=nibabel.aff2axcodes(nuc.affine), cell_axes=nibabel.aff2axcodes(cell.affine),
        nucleus_sha256=nuc_sha, cell_sha256=cell_sha)

for sample in ('Sample18','Sample19','Sample20'):
    try:
        cells=frames(listing(labels[sample+'LabelUnified']),sample,'segCell')
        nuclei=frames(listing(listing(raw[sample])['SegNuc']),sample,'segNuc')
        paired=set(cells)&set(nuclei)
        starts=sorted(t for t in paired if t+1 in paired)
        assert starts, f'No adjacent volumes for {sample}'
        selected=[starts[i] for i in np.linspace(0,len(starts)-1,3,dtype=int)]
        checks=[]
        for t in selected:
            a, before=match_frame(nuclei[t],cells[t])
            b, after=match_frame(nuclei[t+1],cells[t+1])
            checks.append(dict(t=t,before=before,after=after,
                shared_names=len(set(a.values())&set(b.values())),
                shared_names_same_nucleus_id=sum(b.get(k)==v for k,v in a.items())))
        report['samples'][sample]=dict(available_adjacent_pairs=len(starts),checks=checks)
    except (KeyError,ValueError,RuntimeError,OSError,AssertionError) as exc:
        report['samples'][sample]={'error':str(exc)}
    output.write_text(json.dumps(report,indent=2))
    print(sample,report['samples'][sample],flush=True)
